1. **Ты обучил пять разных моделей на абсолютно одних и тех же обучающих данных, и все они дают precision 95%. Есть ли шанс, что объединив эти модели, можно получить лучший результат? Если да — как? Если нет — почему?**

2. **В чём разница между классификаторами с жёстким (hard voting) и мягким (soft voting) голосованием?**

3. **Можно ли ускорить обучение bagging-ансамбля, распределив его по нескольким серверам? А pasting-ансамбля, boosting-ансамбля, случайного леса (random forest) или stacking-ансамбля?**

4. **В чём польза оценки out-of-bag (OOB)?**

5. **Что делает ансамбли extra-trees более случайными, чем обычные случайные леса? Как эта дополнительная случайность помогает? Классификаторы extra-trees медленнее или быстрее обычных случайных лесов?**

6. **Если твой AdaBoost-ансамбль недообучается (underfits) на обучающих данных, какие гиперпараметры стоит покрутить и в какую сторону?**

7. **Если твой gradient boosting ансамбль переобучается (overfits) на обучающей выборке, learning rate нужно увеличить или уменьшить?**

8. **Загрузи датасет MNIST (из главы 3) и раздели его на обучающую, валидационную и тестовую выборки (например, 50 000 экземпляров на обучение, 10 000 на валидацию и 10 000 на тест). Затем обучи разные классификаторы: случайный лес, extra-trees и SVM. Дальше попробуй объединить их в ансамбль, который на валидационной выборке превосходит каждый отдельный классификатор, используя soft или hard voting. Когда такой ансамбль найдётся — проверь его на тестовой выборке. Насколько он лучше каждого отдельного классификатора?**

9. **Прогони отдельные классификаторы из предыдущего упражнения, чтобы получить предсказания на валидационной выборке, и создай из этих предсказаний новую обучающую выборку: каждый экземпляр — это вектор из предсказаний всех твоих классификаторов для одного изображения, а целевое значение — класс этого изображения. Обучи классификатор на этой новой выборке. Поздравляем — ты обучил блендер (blender), и вместе с классификаторами он образует stacking-ансамбль! Теперь оцени ансамбль на тестовой выборке. Для каждого изображения из теста сделай предсказания всеми классификаторами, затем подай эти предсказания в блендер, чтобы получить предсказание ансамбля. Как это сравнить с voting-классификатором, который ты обучил раньше? Теперь попробуй то же самое с `StackingClassifier`. Получается лучше? Если да — почему?**

In [ ]:
1) Да, шанс есть — но только если модели разные (ошибаются на разных объектах).
Если ошибки совпадают — объединение не даст ничего.
И voting, и stacking работают; решает не метод, а разнообразие моделей


In [ ]:
2) Hard voting: каждая модель голосует классом (0/1), побеждает большинство, 
все голоса равны.
Soft voting: каждая модель даёт вероятность (predict_proba), вероятности 
усредняются, берётся argmax.
Разница: soft учитывает уверенность (0.9 весит больше 0.51) -> обычно точнее.
Нужны вероятности у всех моделей: SVC(probability=True), иначе predict_proba нет.

In [ ]:
3) 
Bagging -> ДА. Каждая модель учится на своей bootstrap-выборке 
(с возвращением, размер как у трейна, ~63% уникальных) и не знает про остальные.
Pasting -> ДА, по той же причине (выборка без возвращения, просто часть трейна).
Random Forest -> ДА. Это bagging деревьев, модели независимы.
Boosting (AdaBoost, GBRT) -> НЕТ. Каждое следующее дерево строится на 
ошибках/остатках предыдущего, поэтому обучение строго последовательное: 
распределение по серверам не даёт выигрыша вообще.
Stacking -> ПО СЛОЯМ. Внутри одного слоя модели независимы -> учатся параллельно.
 Но слой можно учить только после того, как готов предыдущий 
 (блендеру нужны предсказания базовых моделей). То есть ускорение есть, но между
  слоями — жёсткая очередь.

В коде: n_jobs=-1 (и parallel_backend/joblib, если реально несколько машин).


In [ ]:
4. 
OOB (out-of-bag) — оценка ансамбля на точках, которые модель не видела.
При bagging каждое дерево учится на выборке с возвратом, поэтому ~37% точек 
остаются вне его мешка. Вот на них оно и проверяется.
Польза: это бесплатный validation. Отдельный validation-набор не нужен, все 
данные остаются в трейне, поэтому ансамбль получается чуть лучше.
Работает у bagging-подобных: bagging, pasting, Random Forest, Extra-Trees. 
В boosting такого нет.
В sklearn: oob_score=True, потом oob_score_.

In [ ]:
5.
Extra-Trees (ET) — те же случайные леса, но с дополнительным рандомом.
Разница с RF: в RF на каждом узле перебираются все признаки из случайного 
подмножества и ищется лучший порог по Gini. В ET признак выбирается случайно 
и порог тоже берётся случайно, без перебора. Один бросок вместо перебора.
Зачем: деревья получаются разные, их ошибки меньше совпадают (декорреляция). 
Каждое дерево по отдельности чуть хуже — bias выше, но разброс ансамбля 
меньше — variance ниже. 
Это регуляризация: если RF переобучается, ET может дать лучше.
Скорость: обучение намного быстрее, потому что нет поиска порога.
 На предсказании скорость такая же, как у RF — пороги уже в узлах, просто 
 идём по дереву вниз.
В sklearn: ExtraTreesClassifier, по умолчанию bootstrap=False (в RF True).*

In [ ]:
6) 
AdaBoost недообучается — значит модель слишком слабая, надо усилить.
Крутить в сторону больше:
n_estimators вверх (больше деревьев)
learning_rate чуть вверх (каждое дерево громче влияет)
регуляризацию базовой модели ослабить (max_depth вверх, если пни слишком тупые)
Правило: недообучение = усиливаем, переобучение = ослабляем.
learning_rate в AdaBoost — это shrinkage, насколько сильно каждое следующее 
дерево влияет на ответ. Маленький = тихо шепчет, ансамбль недобирает.


In [ ]:
7)
 Gradient Boosting переобучается — уменьшаем learning_rate.
learning_rate — размер шага. Большой шаг: каждое дерево сильно влияет, 
ансамбль быстро подгоняет трейн, обобщение плохое. Маленький шаг: мелкие поправки,
обобщение лучше.
Второй рычаг — early stopping: деревьев скорее всего слишком много. 
n_iter_no_change останавливает обучение, когда validation loss перестал 
улучшаться N итераций подряд.
Цена маленького rate: нужно больше деревьев, обучение дольше.